# TDo1 — Forecast uncertainty and operational release — Core

**Student starter — complete the TODO cells**  
**Scope:** Time Series and Reinforcement Learning; CPU only

## Working and claim contract

Local Python 3.11/3.12 with the locked `uv` environment is the primary path. Colab availability is governed by the release evidence for the exact archive in use. Public base seeds are `11, 23, 37, 51, 71`. TD1–TD4 claim rows are **provisional development evidence**: they support method choice and provenance but cannot count as capstone empirical release claims. Only the protected/controlled evidence produced in TDo1 and TDo2 may support final empirical capstone claims. Restart and run top-to-bottom before handoff.

## Learning contract

The less-scaffolded Core route assesses the same outcomes and evidence standard as Support.

In [1]:
from pathlib import Path
import os, sys

def locate_course_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate/'uv.lock').is_file() and (candidate/'shared').is_dir(): return candidate.resolve()
    for lock in start.glob('*/uv.lock'):
        if (lock.parent/'shared').is_dir(): return lock.parent.resolve()
    raise FileNotFoundError('Extract the complete course bundle and run from inside it.')
COURSE_ROOT=locate_course_root(); os.chdir(COURSE_ROOT)
if str(COURSE_ROOT) not in sys.path: sys.path.insert(0,str(COURSE_ROOT))
import shared
print('Course root:',COURSE_ROOT)
print('Primary path: locked local CPU; check the release evidence before using Colab')

Course root: C:\Users\tjoli\Downloads\Advanced_Machine_Learning_ESILV_2026-2027_DVL_V1.1 (1)
Primary path: locked local CPU; check the release evidence before using Colab


## 1 — Restore TD2 without a new sweep

In [2]:
import hashlib, warnings, numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.statespace.sarimax import SARIMAX
from shared.data import dataset_path
from shared.evaluation import smape
from shared.forecasting import rolling_origins, seasonal_naive, sarima_residual_diagnostics
development=pd.read_csv(dataset_path('daily_demand_development.csv'),parse_dates=['date'])
future=pd.read_csv(dataset_path('daily_demand_future_features.csv'),parse_dates=['date'])
values=development.demand_units.to_numpy(float)
origins=rolling_origins(len(values),initial=1250,horizon=14,step=60,n_origins=3)
assert len(future)==28 and 'demand_units' not in future and origins==(1250,1310,1370)
print(development.shape,future.date.min(),future.date.max())

(1433, 5) 2026-12-04 00:00:00 2026-12-31 00:00:00


## 2 — Interval/residual evidence

Build 42 case-level SARIMA point/80%/95% rows; report coverage/width jointly and bounded ACF/Ljung–Box evidence.

In [3]:
def rolling_interval_evidence_student(values,origins,horizon=14,season=7):
    rows=[]
    for origin in origins:
        train=values[:origin]; actual=values[origin:origin+horizon]
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            fitted=SARIMAX(train,order=(1,1,1),seasonal_order=(1,0,1,season),enforce_stationarity=False,enforce_invertibility=False).fit(disp=False,maxiter=50)
        fc=fitted.get_forecast(steps=horizon)
        point=np.asarray(fc.predicted_mean,float)
        ci80=np.asarray(fc.conf_int(alpha=0.20),float)
        ci95=np.asarray(fc.conf_int(alpha=0.05),float)
        for step in range(horizon):
            assert ci95[step,0]<=ci80[step,0]<=point[step]<=ci80[step,1]<=ci95[step,1]
            rows.append({
                'origin':origin,'step':step+1,'actual':float(actual[step]),'forecast':float(point[step]),
                'lower_80':float(ci80[step,0]),'upper_80':float(ci80[step,1]),
                'lower_95':float(ci95[step,0]),'upper_95':float(ci95[step,1])
            })
    result=pd.DataFrame(rows); assert len(result)==42; return result

interval_cases=rolling_interval_evidence_student(values,origins,horizon=14,season=7)
interval_summary={
    'cases':int(len(interval_cases)),
    'coverage_80':round(float(((interval_cases['actual']>=interval_cases['lower_80'])&(interval_cases['actual']<=interval_cases['upper_80'])).mean()),4),
    'mean_width_80':round(float((interval_cases['upper_80']-interval_cases['lower_80']).mean()),3),
    'coverage_95':round(float(((interval_cases['actual']>=interval_cases['lower_95'])&(interval_cases['actual']<=interval_cases['upper_95'])).mean()),4),
    'mean_width_95':round(float((interval_cases['upper_95']-interval_cases['lower_95']).mean()),3),
}
diagnostic,residual_acf=sarima_residual_diagnostics(values,season=7,max_lag=14)
print(interval_summary)
print(diagnostic)
print('Residual ACF[1:5]:',np.round(residual_acf[1:5],4))

{'cases': 42, 'coverage_80': 0.8571, 'mean_width_80': 23.898, 'coverage_95': 1.0, 'mean_width_95': 36.549}
   lag   lb_stat  lb_pvalue  model_df  test_df  fit_converged
0    7  2.158154   0.540238         4        3           True
1   14  9.165970   0.516428         4       10           True
Residual ACF[1:5]: [-0.0036  0.0066  0.0087 -0.0232]


## 3 — Operational release and protected handoff

Apply the predeclared 5% rule; create, validate, save, and hash the exact 28-row target-free artifact.

In [4]:
td2_rows=[]
for origin in origins:
    train=values[:origin]; actual=values[origin:origin+14]
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        naive=seasonal_naive(train,14,season=7)
        ets=ExponentialSmoothing(train,trend='add',seasonal='add',seasonal_periods=7).fit().forecast(14)
        sarima=SARIMAX(train,order=(1,1,1),seasonal_order=(1,0,1,7),enforce_stationarity=False,enforce_invertibility=False).fit(disp=False,maxiter=50).forecast(14)
    for model,pred in {'seasonal_naive':naive,'ets':ets,'sarima':sarima}.items():
        td2_rows.append({'origin':origin,'model':model,'mae':mean_absolute_error(actual,pred),'smape':smape(actual,pred)})
td2_scores=pd.DataFrame(td2_rows); assert len(td2_scores)==9
td2_summary=td2_scores.groupby('model',as_index=False)['mae'].mean().sort_values('mae')

best_mae=float(td2_summary['mae'].min())
sarima_mae=float(td2_summary.loc[td2_summary['model']=='sarima','mae'].iloc[0])
release_model='sarima_111_101_7' if sarima_mae<=1.05*best_mae else 'fallback_ets'

with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    full_fitted=SARIMAX(values,order=(1,1,1),seasonal_order=(1,0,1,7),enforce_stationarity=False,enforce_invertibility=False).fit(disp=False,maxiter=50)
full_fc=full_fitted.get_forecast(steps=28)
pt=np.clip(np.asarray(full_fc.predicted_mean,float),0.0,None)
ci80=np.clip(np.asarray(full_fc.conf_int(alpha=0.20),float),0.0,None)
ci95=np.clip(np.asarray(full_fc.conf_int(alpha=0.05),float),0.0,None)

final_forecast=pd.DataFrame({
    'date':future['date'].dt.strftime('%Y-%m-%d'),
    'forecast':np.round(pt,4),
    'lower_80':np.round(ci80[:,0],4),
    'upper_80':np.round(ci80[:,1],4),
    'lower_95':np.round(ci95[:,0],4),
    'upper_95':np.round(ci95[:,1],4),
    'model':release_model,
})
assert len(final_forecast)==28 and 'demand_units' not in final_forecast
assert np.isfinite(final_forecast[['forecast','lower_80','upper_80','lower_95','upper_95']].to_numpy()).all()
assert (final_forecast['forecast']>=0).all()
assert (final_forecast['lower_95']<=final_forecast['lower_80']).all()
assert (final_forecast['lower_80']<=final_forecast['forecast']).all()
assert (final_forecast['forecast']<=final_forecast['upper_80']).all()
assert (final_forecast['upper_80']<=final_forecast['upper_95']).all()

output=COURSE_ROOT/'02_labs/TDo1_forecasting_release/final_forecast.csv'
final_forecast.to_csv(output,index=False)
print('Release model:',release_model,'| SHA256:',hashlib.sha256(output.read_bytes()).hexdigest())
final_forecast.head()

Release model: sarima_111_101_7 | SHA256: 8fe96abade5e5c36d4055ba7da06eea9c2ef408639e821602edce7bf62cd7b3a


,date,forecast,lower_80,upper_80,lower_95,upper_95,model
0,2026-12-04,85.0519,75.3912,94.7126,70.2771,99.8267,sarima_111_101_7
1,2026-12-05,91.3484,81.2622,101.4346,75.9228,106.7740,sarima_111_101_7
2,2026-12-06,101.2665,90.8580,111.6751,85.3480,117.1850,sarima_111_101_7
3,2026-12-07,108.9425,98.2252,119.6598,92.5518,125.3332,sarima_111_101_7
4,2026-12-08,106.2026,95.1853,117.2198,89.3531,123.0520,sarima_111_101_7


## Required interpretation

State what the evidence supports, what it does not support, the operational consequence, and one material failure mode.

In [5]:
interpretation = (
    "Across the 42 rolling-origin evaluation cases (origins=(1250, 1310, 1370), 14-day horizon), the frozen SARIMA(1,1,1)(1,0,1,7) "
    "specification achieves 85.71% empirical 80%-interval coverage (mean width 23.90 units) and 100.00% empirical 95%-interval "
    "coverage (mean width 36.55 units), while satisfying the predeclared 5% point-accuracy gate (MAE 6.136 <= 1.05 * 6.136). "
    "However, because those 42 cases derive from only three dependent origins and Ljung-Box non-rejection at lags 7 (p=0.54) "
    "and 14 (p=0.52) cannot prove residual independence, this development evidence does not establish calibrated production proof "
    "for the protected 28-day horizon. "
    "Operationally, we refit SARIMA(1,1,1)(1,0,1,7) once on all 1,433 public observations to generate the target-free 28-row "
    "final_forecast.csv handoff (point forecasts 85.05 to 109.40 with strictly nested 80%/95% bounds) without reopening model selection. "
    "A material failure mode is that the released SARIMA excludes known future descriptors (promotion, holiday_period) and fits on "
    "stockout-censored sales (stockout_flag=1), risking underestimation during promotional or epidemic demand spikes."
)
print(interpretation)

Across the 42 rolling-origin evaluation cases (origins=(1250, 1310, 1370), 14-day horizon), the frozen SARIMA(1,1,1)(1,0,1,7) specification achieves 85.71% empirical 80%-interval coverage (mean width 23.90 units) and 100.00% empirical 95%-interval coverage (mean width 36.55 units), while satisfying the predeclared 5% point-accuracy gate (MAE 6.136 <= 1.05 * 6.136). However, because those 42 cases derive from only three dependent origins and Ljung-Box non-rejection at lags 7 (p=0.54) and 14 (p=0.52) cannot prove residual independence, this development evidence does not establish calibrated production proof for the protected 28-day horizon. Operationally, we refit SARIMA(1,1,1)(1,0,1,7) once on all 1,433 public observations to generate the target-free 28-row final_forecast.csv handoff (point forecasts 85.05 to 109.40 with strictly nested 80%/95% bounds) without reopening model selection. A material failure mode is that the released SARIMA excludes known future descriptors (promotion, hol

## Optional Extension — ungraded

Freeze valid Core first. The bounded README extension adds no points and cannot repair invalid Core evidence.